# AI Battery Safety Guardian — M8 Incident Classifier (DistilBERT Fine-Tuning)

This notebook fine-tunes `distilbert-base-uncased` on synthetic technician/driver battery incident notes across 5 fault categories and compares against a TF-IDF + Logistic Regression baseline on a Google Colab **NVIDIA T4 GPU**.

### Pipeline Structure:
- **Cell 1**: GPU Check & Validation
- **Cell 2**: Upload `colab_pack.zip` & Install Dependencies
- **Cell 3**: Fast Smoke Test
- **Cell 4**: Full 3-Epoch Training Run & Baseline Comparison
- **Cell 5**: Bundle `results_bert.zip` & Auto-Download

In [ ]:
# Cell 1: Check GPU environment
import torch

if not torch.cuda.is_available():
    raise SystemError("❌ GPU not detected! In Google Colab, go to 'Runtime' -> 'Change runtime type' and select 'T4 GPU'.")

gpu_name = torch.cuda.get_device_name(0)
print(f"✅ Active GPU: {gpu_name}")
print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
print(f"Allocated VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Cell 2: Upload colab_pack.zip and install dependencies
import os
import zipfile
from google.colab import files

if not os.path.exists('colab_pack.zip'):
    print("Please select and upload 'colab_pack.zip' from your local project:")
    uploaded = files.upload()

print("Extracting colab_pack.zip...")
with zipfile.ZipFile('colab_pack.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("Installing dependencies from requirements-colab.txt...")
!pip uninstall -y -q torchao
!pip install -q -r requirements-colab.txt
print("✅ Setup complete!")

In [ ]:
# Cell 3: Fast smoke test (20 samples per class, 1 epoch)
print("Running smoke test to verify execution pipeline...")
!python scripts/train_bert.py \
    --data data/incidents/incidents.csv \
    --smoke \
    --out-dir artifacts/models/incident_classifier_smoke \
    --metrics-out artifacts/metrics/bert_smoke.json \
    --cm-out artifacts/figures/bert_cm_smoke.png

print("✅ Smoke test succeeded without errors!")

In [ ]:
# Cell 4: Full training run (3 epochs, stratified 70/15/15 split, baseline comparison)
print("Starting full DistilBERT fine-tuning + TF-IDF baseline...")
!python scripts/train_bert.py \
    --data data/incidents/incidents.csv \
    --epochs 3 \
    --batch-size 16 \
    --out-dir artifacts/models/incident_classifier \
    --metrics-out artifacts/metrics/bert.json \
    --cm-out artifacts/figures/bert_confusion_matrix.png

import json
with open('artifacts/metrics/bert.json') as f:
    metrics = json.load(f)

print("\n--- EVALUATION SUMMARY ---")
print(f"Baseline TF-IDF Macro-F1: {metrics['baseline_tfidf']['macro_f1']:.4f}")
print(f"DistilBERT Macro-F1:     {metrics['distilbert']['macro_f1']:.4f}")
print(f"DistilBERT Accuracy:     {metrics['distilbert']['accuracy']:.1%}")
print("Per-class F1 Scores:")
for cat, f1 in metrics['distilbert']['per_class_f1'].items():
    print(f"  - {cat}: {f1:.4f}")

In [ ]:
# Cell 5: Create results bundle and auto-download
import shutil
from google.colab import files

bundle_name = 'results_bert.zip'
export_dir = 'export_bert'
os.makedirs(f'{export_dir}/artifacts/models/incident_classifier', exist_ok=True)
os.makedirs(f'{export_dir}/artifacts/metrics', exist_ok=True)
os.makedirs(f'{export_dir}/artifacts/figures', exist_ok=True)

# Copy model artifacts
shutil.copytree('artifacts/models/incident_classifier', f'{export_dir}/artifacts/models/incident_classifier', dirs_exist_ok=True)
# Copy metrics
shutil.copy2('artifacts/metrics/bert.json', f'{export_dir}/artifacts/metrics/bert.json')
# Copy confusion matrix plot
if os.path.exists('artifacts/figures/bert_confusion_matrix.png'):
    shutil.copy2('artifacts/figures/bert_confusion_matrix.png', f'{export_dir}/artifacts/figures/bert_confusion_matrix.png')

# Create zip
shutil.make_archive('results_bert', 'zip', export_dir)
print(f"✅ Bundle '{bundle_name}' created successfully ({os.path.getsize(bundle_name) / 1e6:.2f} MB).")
print("Downloading results bundle to your local machine...")
files.download(bundle_name)